# 01 — Worker

**Run this unedited, every session.** It claims whatever the ledger says is
next and runs it.

There is nothing to configure. The cell, scene and seed come from the ledger,
which enforces stage order and prerequisites globally — which is precisely why
this is one notebook rather than eight.

If a session dies mid-run, do nothing: the row is left with a stale heartbeat
and the next session reclaims it. Interrupted runs **restart** rather than
resume, deliberately — the checkpoint omits the medium model, the codebooks and
the loop's schedule flags, so resuming would silently reinitialise β and
produce a run that looks complete and is a different experiment.


## 1. Drive and paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# ---------------------------------------------------------------------------
# The one place paths are defined. Everything else derives from DRIVE_ROOT.
#
#   e3dgsuw/
#     dataset/     the four scenes (original) + undistorted/  <- created below
#     dense/       M1 clouds, with SHA-256 sidecars
#     runs/        <cell>/<scene>/s<seed>/  -- one run, all of it together
#     analysis/    analyse.py output, figures, tables
#     run_ledger.json
# ---------------------------------------------------------------------------
DRIVE_ROOT   = '/content/drive/MyDrive/e3dgsuw'
DATASET_DIR  = f'{DRIVE_ROOT}/dataset'
DATA_UNDIST  = f'{DATASET_DIR}/undistorted'
DENSE_DIR    = f'{DRIVE_ROOT}/dense'
ANALYSIS_DIR = f'{DRIVE_ROOT}/analysis'

# Training reads from local disk, not Drive: the scene loader pulls every image
# at startup, and Drive's FUSE layer makes that far slower than a single copy.
LOCAL_DATA   = '/content/data'

REPO_URL  = 'https://github.com/dinanirham/An-Efficient-3D-Gaussian-Splatting-for-Underwater-3D-Reconstruction.git'
REPO_DIR  = '/content/e3dgsuw'
IMPL_DIR  = f'{REPO_DIR}/implementation'
SCENES    = ['Curasao', 'IUI3-RedSea', 'JapaneseGradens-RedSea', 'Panama']

import os
assert os.path.isdir(DRIVE_ROOT), (
    f'{DRIVE_ROOT} not found. Check the folder name, or edit DRIVE_ROOT above.')
for d in (DATA_UNDIST, DENSE_DIR, f'{DRIVE_ROOT}/runs', ANALYSIS_DIR):
    os.makedirs(d, exist_ok=True)

# Export them so the `!` cells below resolve "$DRIVE_ROOT" as a real shell
# variable. Relying on IPython to substitute notebook variables into magics
# works until it doesn't, and when it doesn't it substitutes nothing and the
# command runs against a silently truncated path rather than failing.
os.environ.update(
    DRIVE_ROOT=DRIVE_ROOT, DATASET_DIR=DATASET_DIR, DATA_UNDIST=DATA_UNDIST,
    DENSE_DIR=DENSE_DIR, ANALYSIS_DIR=ANALYSIS_DIR, LOCAL_DATA=LOCAL_DATA,
    REPO_DIR=REPO_DIR, IMPL_DIR=IMPL_DIR,
)


def find_originals():
    """Locate the four scenes under dataset/, however they were arranged.

    Accepts the scenes directly under dataset/, or nested one level (e.g.
    dataset/SeathruNeRF_dataset/). Returns the directory that contains them.
    """
    candidates = [DATASET_DIR] + [
        os.path.join(DATASET_DIR, d) for d in sorted(os.listdir(DATASET_DIR))
        if os.path.isdir(os.path.join(DATASET_DIR, d)) and d != 'undistorted'
    ]
    for base in candidates:
        if all(os.path.isdir(os.path.join(base, s)) for s in SCENES):
            return base
    return None


DATA_ORIG = find_originals()
print('drive root :', DRIVE_ROOT)
print('originals  :', DATA_ORIG or 'NOT FOUND')
print('undistorted:', DATA_UNDIST)


Mounted at /content/drive
drive root : /content/drive/MyDrive/e3dgsuw
originals  : /content/drive/MyDrive/e3dgsuw/dataset/SeaThruNeRF_dataset
undistorted: /content/drive/MyDrive/e3dgsuw/dataset/undistorted


## 2. GPU — must be an A100

In [2]:
import subprocess, torch
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout)
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
cap  = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)
print(f'torch {torch.__version__}  cuda {torch.version.cuda}  {name}  sm_{cap[0]}{cap[1]}')

# Every conclusion in this study is a between-cell contrast, and cells on
# different devices are not comparable. Stop now rather than produce a run
# that has to be discarded later.
assert 'A100' in name, f'Expected an A100, got {name!r}. Restart the runtime.'


NVIDIA A100-SXM4-40GB, 40960 MiB, 580.82.07

torch 2.11.0+cu128  cuda 12.8  NVIDIA A100-SXM4-40GB  sm_80


## 3. Clone and build  *(a few minutes)*

In [5]:
import os, subprocess

# Private repo? Add a Colab secret named GITHUB_TOKEN (key icon in the left
# sidebar) with a fine-grained read token, and toggle notebook access on.
# Read from Secrets rather than pasted into the cell: a pasted token is saved
# inside the .ipynb, which then travels wherever the notebook does.
GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN') or None
    print('GITHUB_TOKEN: loaded from Colab Secrets')
except ImportError:
    pass                                  # not running under Colab
except Exception as e:                    # secret absent, or access not granted
    print(f'GITHUB_TOKEN: not available ({type(e).__name__}) -- '
          'fine for a public repo')

url = REPO_URL
if GITHUB_TOKEN:
    url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')

# Never let git fall back to an interactive credential prompt: in a notebook it
# hangs the cell indefinitely with nothing on screen to say why.
env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}


def _redact(s):
    """Strip the token from git output -- git echoes the remote URL on failure,
    and notebook outputs are saved to the file and shared with it."""
    return s.replace(GITHUB_TOKEN, '***') if GITHUB_TOKEN else s


if os.path.isdir(REPO_DIR) and not os.path.isdir(f'{REPO_DIR}/.git'):
    raise RuntimeError(
        f'{REPO_DIR} exists but is not a git checkout -- probably a clone that '
        f'died partway. Delete it and re-run this cell.')

if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git','clone','--depth','1',url,REPO_DIR],
                       capture_output=True, text=True, env=env)
    if r.returncode != 0:
        raise RuntimeError(
            'clone failed. If the repository is private, add a GITHUB_TOKEN '
            'secret in Colab and grant this notebook access.\n'
            f'{_redact(r.stderr)[-800:]}')
else:
    # Repoint the remote before pulling. The stored URL was written by an
    # earlier clone, which may have run without a token (or with a stale one);
    # injecting the token into `url` alone never reaches the pull.
    subprocess.run(['git','-C',REPO_DIR,'remote','set-url','origin',url],
                   check=True, env=env)
    r = subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],
                       capture_output=True, text=True, env=env)
    if r.returncode != 0:
        raise RuntimeError(
            'pull failed. If the repository is private, check the GITHUB_TOKEN '
            'secret is set and this notebook has access.\n'
            f'{_redact(r.stderr)[-800:]}')

# Fail here, naming the directory, rather than letting a later cell run from
# whatever the working directory happened to be.
assert os.path.isdir(IMPL_DIR), (
    f'clone produced no {IMPL_DIR}. Contents of {REPO_DIR}: '
    f'{sorted(os.listdir(REPO_DIR)) if os.path.isdir(REPO_DIR) else "missing"}')

os.chdir(IMPL_DIR)
print(subprocess.run(['git','-C',REPO_DIR,'log','--oneline','-1'],
                     capture_output=True, text=True).stdout.strip())
print('cwd:', os.getcwd())


GITHUB_TOKEN: loaded from Colab Secrets
f382f05 Merge pull request #7 from dinanirham/fix/romatch-and-silent-dense-failure
cwd: /content/e3dgsuw/implementation


In [6]:
# Builds diff_gaussian_rasterization_ms and simple_knn against whatever torch
# Colab ships -- deliberately NOT installing our own, which would risk a
# mismatch between torch's CUDA and the toolkit the extensions compile with.
# Takes a few minutes; must be repeated each session.
#
# chdir explicitly rather than via `%cd $IMPL_DIR`: a magic whose variable fails
# to expand reports the *current* directory and continues, so the build then
# runs from the wrong place and fails two steps later with a bare
# "tools/setup_colab.sh: No such file or directory".
import os
assert os.path.isdir(IMPL_DIR), (
    f'{IMPL_DIR} not found -- run the "Clone the repository" cell above first.')
os.chdir(IMPL_DIR)
print('building in', os.getcwd())
!bash tools/setup_colab.sh


building in /content/e3dgsuw/implementation
 repo root        : /content/e3dgsuw
 implementation   : /content/e3dgsuw/implementation
 arch list        : 8.0+PTX

--- GPU ---
NVIDIA A100-SXM4-40GB, 40960 MiB, 580.82.07

--- sanity: the reference submodules must be present ---
  ok: /content/e3dgsuw/mini-splatting/submodules/diff-gaussian-rasterization_ms
  ok: /content/e3dgsuw/mini-splatting/submodules/simple-knn
  ok: /content/e3dgsuw/mini-splatting/submodules/diff-gaussian-rasterization_ms/third_party/glm

--- python deps Colab does not ship ---
--- RoMa (M1 only) ---
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
             source/roma_init.py will not run. A0/A2/A3/A6 are fine.

--- building diff_gaussian_rasterization_ms (Mini-Splatting fork) ---
    ok
--- building simple_knn ---
    ok

--- import check ---
torch 2.11.0+cu128  cuda=True
device: NVIDIA A100-SXM4-40GB  sm_80
rasterizer: /usr/l

In [7]:
import importlib, torch
for m in ('diff_gaussian_rasterization_ms', 'simple_knn'):
    importlib.import_module(m)
print('extensions import OK  |  torch', torch.__version__,
      '| cuda', torch.version.cuda)


extensions import OK  |  torch 2.11.0+cu128 | cuda 12.8


## 4. Verify the rasterizer

In [8]:
!python -m tools.verify_rasterizer


[PASS] T0 extension builds and targets sm_80+: NVIDIA A100-SXM4-40GB sm_80, module at /usr/local/lib/python3.13/dist-packages/diff_gaussian_rasterization_ms/__init__.py
[PASS] T1 alpha in [0,1] and monotone in opacity: alpha_hi_max=0.8846 alpha_lo_max=0.000000
[PASS] T2 alpha composites as 1-(1-a1)(1-a2): centre: one=0.4467 two=0.6939 expected=0.6939
[PASS] T3 Z_raw/alpha recovers true depth  <-- decisive: o=0.3: max|Z/a - z|=0.00e+00; o=0.7: max|Z/a - z|=0.00e+00; o=0.95: max|Z/a - z|=2.38e-07
[PASS] T4 alpha is differentiable w.r.t. opacity: d(sum alpha)/d(opacity_logit) = [3.4596166610717773]
[PASS] T5 importance accumulators present and sane: {'accum_weights': (2,), 'area_proj': (2,), 'area_max': (2,)}, area_max_sum=118.0
[PASS] T6 zero background does not leak into probe: corner alpha=0.00e+00 depth=0.00e+00 (both must be ~0)

M1 ACCEPTANCE: PASSED (7/7)


## 5. Stage the dataset locally

The loader reads every image at startup; from Drive that is markedly slower
than one bulk copy.


In [9]:
import os, shutil, time
missing = [s for s in SCENES if not os.path.isdir(f'{DATA_UNDIST}/{s}')]
assert not missing, (
    f'Undistorted scenes missing: {missing}. Run 00_setup.ipynb first — the '
    f'scenes are OPENCV-model and will not load undistorted.')

os.makedirs(LOCAL_DATA, exist_ok=True)
t0 = time.time()
for s in SCENES:
    if not os.path.exists(f'{LOCAL_DATA}/{s}'):
        shutil.copytree(f'{DATA_UNDIST}/{s}', f'{LOCAL_DATA}/{s}')
print(f'staged in {time.time()-t0:.0f}s')
!python -m tools.run_ledger status --output_root "$DRIVE_ROOT"


staged in 32s
ledger : /content/drive/MyDrive/e3dgsuw/run_ledger.json
budget : NOT SET (blocks all m2 cells)
total  : 96 runs  pending=96

  S1 (A0          ) pending=12
  S2 (A2          ) pending=12
  S3 (A1,A3       ) pending=24
  S4 (A4,A5,A6    ) pending=36
  S5 (A7          ) pending=12


## 6. Work

`--max_minutes` sits **below** the session limit so the loop stops claiming new
runs and exits cleanly rather than being killed mid-run. Raise it if your
sessions run longer.

Until the budget is set, every M2 cell is blocked and the queue says so — that
is expected during S1.


In [10]:
!python -m tools.run_queue \
    --output_root "$DRIVE_ROOT" \
    --data_root   "$LOCAL_DATA" \
    --max_minutes 200


[queue] gpu: NVIDIA A100-SXM4-40GB
[queue] A0/Curasao/s0: cannot build command: no images_wb directory under /content/data/Curasao
[queue] A0/Curasao/s0: cannot build command: no images_wb directory under /content/data/Curasao
[queue] A0/Curasao/s0: cannot build command: no images_wb directory under /content/data/Curasao
[queue] A0/Curasao/s1: cannot build command: no images_wb directory under /content/data/Curasao
[queue] A0/Curasao/s1: cannot build command: no images_wb directory under /content/data/Curasao
[queue] A0/Curasao/s1: cannot build command: no images_wb directory under /content/data/Curasao
[queue] A0/Curasao/s2: cannot build command: no images_wb directory under /content/data/Curasao
[queue] A0/Curasao/s2: cannot build command: no images_wb directory under /content/data/Curasao
[queue] A0/Curasao/s2: cannot build command: no images_wb directory under /content/data/Curasao
[queue] A0/IUI3-RedSea/s0: cannot build command: no images_wb directory under /content/data/IUI3-RedS

## 7. After S1 (A0) completes — set the budget

The primitive budget comes from A0's converged count, which no publication of
the baseline reports. Until it is set, A2, A4, A6 and A7 stay blocked.

Pick a value **below** the counts below so the budget actually binds. A budget
that does not bind makes A4 equivalent to A1 and A7 to A5, and a null
interaction measured in that state is a configuration artifact, not a finding.


In [ ]:
import glob, csv, statistics
counts = []
for f in sorted(glob.glob(f'{DRIVE_ROOT}/runs/A0/*/s*/diagnostics.csv')):
    rows = list(csv.DictReader(open(f)))
    if rows:
        counts.append((f.split('/runs/')[1].rsplit('/', 1)[0],
                       int(rows[-1]['n_primitives'])))
for name, n in counts:
    print(f'{n:>12,}  {name}')
if counts:
    med = statistics.median(n for _, n in counts)
    print(f'\nmedian {med:,.0f}   suggested budget ~{int(med*0.6):,} (60%)')
    print('Then run:')
    print(f'  !python -m tools.run_ledger set-budget <count> --output_root "$DRIVE_ROOT"')
else:
    print('No A0 diagnostics yet.')


---
Re-run this notebook in a fresh session to continue. Nothing changes between
sessions.
